# LSTM Autoencoder Simulation — Shredder Bearing Anomaly Detection

## 개요

이 노트북은 **LSTM Autoencoder**를 이용한 이상 탐지(Anomaly Detection)를 체험하는 시뮬레이션입니다.

| 항목 | 내용 |
|------|------|
| **모델** | LSTM Autoencoder (순환 신경망) |
| **시나리오** | 정상 운전 패턴 학습 → 비정상(이상) 자동 감지 |
| **핵심** | 다변량(온도+진동+전류), 시퀀스 패턴 기억, 재구성 오차 기반 탐지 |

### LSTM Autoencoder 원리

```
정상 데이터 → Encoder(압축) → Bottleneck → Decoder(복원)
                                               ↓
              정상이면 복원 잘 됨 → 오차 작음 → 정상 판정
              이상이면 복원 못 함 → 오차 큼  → 이상 판정!
```

### Prophet과의 비교

| 비교 | Prophet | LSTM |
|:---:|:---:|:---:|
| 입력 변수 | 1개 (단변량) | **여러 개 (다변량)** |
| 목적 | 미래 값 예측 | **이상 패턴 감지** |
| 학습 데이터 | 정상+이상 혼합 OK | **정상 데이터만 필요** |
| 해석 | 쉬움 (분해) | 어려움 (블랙박스) |

> **GPU 권장**: 상단 메뉴 → 런타임 → 런타임 유형 변경 → T4 GPU 선택

---
## Step 0. Library Install & Import

In [ ]:
!pip install -q scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, RepeatVector, TimeDistributed

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

---
## Step 1. Shredder Sensor Data Generation

Prophet에서는 온도 1개만 사용했지만, LSTM은 **3개 센서를 동시에** 사용합니다:

| 센서 | 단위 | 역할 |
|------|------|------|
| temperature | °C | 베어링 온도 (주 감시 대상) |
| vibration | mm/s | 진동 RMS (마모/불균형 지표) |
| current | A | 모터 전류 (부하 지표) |

이 3개 센서가 **동시에** 비정상 패턴을 보이면 이상으로 감지합니다.

In [ ]:
def generate_shredder_data(days=90, freq_minutes=10, seed=42):
    """Shredder sensor data simulator."""
    np.random.seed(seed)

    n_points = days * 24 * 60 // freq_minutes
    timestamps = pd.date_range(start='2026-01-01', periods=n_points, freq=f'{freq_minutes}min')

    t = np.arange(n_points)
    hours = np.array([ts.hour for ts in timestamps])
    dow = np.array([ts.dayofweek for ts in timestamps])

    # Bearing Temperature
    base_temp = 28.0
    daily_pattern = 5.0 * np.sin(2 * np.pi * hours / 24 - np.pi/2)
    weekly_pattern = np.where(dow >= 5, -3.0, 0.0)
    wear_trend = 0.03 * t / (24 * 60 / freq_minutes)
    noise = np.random.normal(0, 0.8, n_points)
    anomaly_mask = np.random.random(n_points) < 0.005
    anomaly_spike = anomaly_mask * np.random.uniform(15, 30, n_points)
    temperature = np.clip(base_temp + daily_pattern + weekly_pattern + wear_trend + noise + anomaly_spike, 15, 80)

    # Vibration RMS
    base_vib = 2.5
    vib_daily = 0.5 * np.sin(2 * np.pi * hours / 24)
    vib_wear = 0.02 * t / (24 * 60 / freq_minutes)
    vib_noise = np.random.normal(0, 0.3, n_points)
    vib_anomaly = anomaly_mask * np.random.uniform(5, 15, n_points)
    vibration = np.clip(base_vib + vib_daily + vib_wear + vib_noise + vib_anomaly, 0.5, 25)

    # Motor Current
    base_cur = 85.0
    cur_daily = 10.0 * np.sin(2 * np.pi * hours / 24 - np.pi/3)
    cur_wear = 0.05 * t / (24 * 60 / freq_minutes)
    cur_noise = np.random.normal(0, 2.0, n_points)
    cur_weekend = np.where(dow >= 5, -30.0, 0.0)
    current = np.clip(base_cur + cur_daily + cur_wear + cur_noise + cur_weekend, 20, 150)

    # RPM
    rpm = np.clip(20.0 + np.random.normal(0, 0.3, n_points) + np.where(dow >= 5, -15.0, 0.0), 0, 25)

    # Throughput
    throughput = np.clip(2.5 + 0.5 * np.sin(2*np.pi*hours/24 - np.pi/4) + np.random.normal(0, 0.15, n_points) + np.where(dow >= 5, -2.0, 0.0), 0, 4)

    df = pd.DataFrame({
        'timestamp': timestamps,
        'temperature': np.round(temperature, 2),
        'vibration': np.round(vibration, 2),
        'current': np.round(current, 2),
        'rpm': np.round(rpm, 2),
        'throughput': np.round(throughput, 2),
        'is_anomaly': anomaly_mask  # ground truth for evaluation
    })
    return df

print('generate_shredder_data() defined.')

In [ ]:
# Generate data (1-hour intervals for LSTM)
df = generate_shredder_data(days=90, freq_minutes=60)

features = ['temperature', 'vibration', 'current']
print(f'Generated: {len(df)} samples')
print(f'Period: {df["timestamp"].min()} ~ {df["timestamp"].max()}')
print(f'Features used: {features}')
print(f'Anomaly events: {df["is_anomaly"].sum()} ({df["is_anomaly"].mean()*100:.1f}%)')
print(f'\nStatistics:')
df[features].describe().round(2)

---
## Step 2. Raw Data Visualization

3개 센서 데이터를 함께 확인합니다. **빨간 점**은 이상 이벤트(ground truth)입니다.

**관찰 포인트:**
- 온도와 진동이 동시에 튀는 지점 → 이상 이벤트
- 전류는 부하에 따라 변동하지만 이상 스파이크는 없음

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

sensor_info = [
    ('temperature', 'Bearing Temperature (C)', 'tab:red'),
    ('vibration', 'Vibration RMS (mm/s)', 'tab:blue'),
    ('current', 'Motor Current (A)', 'tab:green'),
]

anomaly_idx = df[df['is_anomaly']].index

for ax, (col, ylabel, color) in zip(axes, sensor_info):
    ax.plot(df['timestamp'], df[col], color=color, alpha=0.6, linewidth=0.5, label='Normal')
    ax.scatter(df.loc[anomaly_idx, 'timestamp'], df.loc[anomaly_idx, col],
               color='red', s=15, zorder=5, label='Anomaly (ground truth)')
    ax.set_ylabel(ylabel, fontsize=11)
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(True, alpha=0.3)

axes[0].set_title('3 Sensor Channels — LSTM uses all simultaneously', fontsize=14, fontweight='bold')
axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

### Sensor Correlation

온도, 진동, 전류 간 상관관계를 확인합니다. 이상 이벤트 시 온도와 진동이 동시에 급등하는 패턴을 확인할 수 있습니다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

pairs = [('temperature', 'vibration'), ('temperature', 'current'), ('vibration', 'current')]
for ax, (x, y) in zip(axes, pairs):
    normal = df[~df['is_anomaly']]
    anomaly = df[df['is_anomaly']]
    ax.scatter(normal[x], normal[y], alpha=0.1, s=3, color='blue', label='Normal')
    ax.scatter(anomaly[x], anomaly[y], alpha=0.8, s=20, color='red', label='Anomaly')
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Sensor Correlations (Red = Anomaly)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 3. Data Preprocessing

LSTM 학습을 위한 전처리:

1. **StandardScaler**: 각 센서를 평균=0, 표준편차=1로 정규화 (LSTM은 스케일에 민감)
2. **Train/Test 시간순 분할**: 70% 학습 / 30% 테스트
3. **시퀀스 변환**: 연속 30시간을 하나의 입력 윈도우로 묶음

```
시퀀스 변환 예시 (window=30):
  입력 1: [t0, t1, t2, ..., t29]   → 3개 센서 × 30시간 = (30, 3)
  입력 2: [t1, t2, t3, ..., t30]
  입력 3: [t2, t3, t4, ..., t31]
  ...
```

In [ ]:
# StandardScaler normalization
data = df[features].values
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

# Time-ordered split
train_size = int(len(data_scaled) * 0.7)
train_data = data_scaled[:train_size]
test_data = data_scaled[train_size:]

print(f'Train: {train_size} samples ({df["timestamp"].iloc[0].date()} ~ {df["timestamp"].iloc[train_size-1].date()})')
print(f'Test : {len(test_data)} samples ({df["timestamp"].iloc[train_size].date()} ~ {df["timestamp"].iloc[-1].date()})')
print(f'\nScaled mean (train): {train_data.mean(axis=0).round(4)}')
print(f'Scaled std  (train): {train_data.std(axis=0).round(4)}')

In [ ]:
def create_sequences(data, seq_length=30):
    """Convert time series to sliding window sequences for LSTM."""
    sequences = []
    for i in range(len(data) - seq_length):
        sequences.append(data[i:i+seq_length])
    return np.array(sequences)

SEQ_LENGTH = 30  # 30-hour window

X_train = create_sequences(train_data, SEQ_LENGTH)
X_test = create_sequences(test_data, SEQ_LENGTH)

print(f'X_train shape: {X_train.shape}  (sequences, timesteps, features)')
print(f'X_test  shape: {X_test.shape}')
print(f'\nEach input: {SEQ_LENGTH} hours x {len(features)} sensors = {SEQ_LENGTH * len(features)} values')

### Sequence 시각화

하나의 입력 시퀀스(30시간 윈도우)가 어떻게 생겼는지 확인합니다.

In [ ]:
# Visualize one sequence
fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)

sample_seq = X_train[0]  # first sequence
for i, (ax, feat) in enumerate(zip(axes, features)):
    ax.plot(range(SEQ_LENGTH), sample_seq[:, i], 'o-', markersize=3, linewidth=1)
    ax.set_ylabel(f'{feat}\n(scaled)')
    ax.grid(True, alpha=0.3)

axes[0].set_title(f'One Input Sequence ({SEQ_LENGTH}-hour window, 3 sensors)', fontsize=13, fontweight='bold')
axes[-1].set_xlabel('Time step (hours)')
plt.tight_layout()
plt.show()

---
## Step 4. LSTM Autoencoder Model

### 모델 구조

```
Input (30, 3)                    ← 30시간 × 3센서
  ↓
LSTM(32) → LSTM(16)              ← Encoder: 압축 (30×3 → 16)
  ↓
RepeatVector(30)                 ← Bottleneck: 핵심 패턴만 유지
  ↓
LSTM(16) → LSTM(32)              ← Decoder: 복원 (16 → 30×3)
  ↓
TimeDistributed(Dense(3))        ← Output (30, 3)
```

**핵심**: 입력과 출력이 **동일** (Autoencoder). 정상 데이터의 패턴을 압축/복원하는 방법을 학습합니다.
이상 데이터가 들어오면 학습하지 못한 패턴이므로 **복원이 잘 안 되고 오차가 커집니다**.

In [ ]:
# Build LSTM Autoencoder
model = Sequential([
    # Encoder: compress (30, 3) -> 16-dim representation
    LSTM(32, input_shape=(SEQ_LENGTH, len(features)), return_sequences=True),
    LSTM(16, return_sequences=False),

    # Bottleneck: only essential patterns survive
    RepeatVector(SEQ_LENGTH),

    # Decoder: reconstruct 16-dim -> (30, 3)
    LSTM(16, return_sequences=True),
    LSTM(32, return_sequences=True),
    TimeDistributed(Dense(len(features)))
])

model.compile(optimizer='adam', loss='mse')
model.summary()

---
## Step 5. Training

**중요**: Autoencoder는 `X_train`을 입력으로도, 출력(타겟)으로도 사용합니다.

`model.fit(X_train, X_train)` → "이 정상 패턴을 기억해!" 

학습이 진행될수록 loss(재구성 오차)가 줄어들면서, 모델이 정상 패턴을 점점 잘 기억하게 됩니다.

In [ ]:
history = model.fit(
    X_train, X_train,       # Input = Output (Autoencoder!)
    epochs=30,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

print(f'\nFinal train loss: {history.history["loss"][-1]:.6f}')
print(f'Final val   loss: {history.history["val_loss"][-1]:.6f}')

### Training Loss Curve

학습 곡선이 수렴하는지 확인합니다. val_loss가 train_loss와 비슷하게 떨어지면 과적합 없이 잘 학습된 것입니다.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history.history['loss'], 'b-', linewidth=2, label='Train Loss')
ax.plot(history.history['val_loss'], 'r--', linewidth=2, label='Val Loss')
ax.set_title('LSTM Autoencoder Training Loss', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Step 6. Anomaly Detection

학습된 모델로 각 시퀀스를 복원하고, **재구성 오차(Reconstruction Error)**를 계산합니다.

- **정상 데이터**: 학습된 패턴과 유사 → 복원 잘 됨 → 오차 작음
- **이상 데이터**: 학습되지 않은 패턴 → 복원 실패 → 오차 큼!

**임계값(Threshold)**: Train 데이터의 95 percentile을 사용합니다.

In [ ]:
# Reconstruction error
train_pred = model.predict(X_train, verbose=0)
test_pred = model.predict(X_test, verbose=0)

train_mse = np.mean(np.power(X_train - train_pred, 2), axis=(1, 2))
test_mse = np.mean(np.power(X_test - test_pred, 2), axis=(1, 2))

# Threshold = 95th percentile of train error
threshold = np.percentile(train_mse, 95)
anomalies = test_mse > threshold

print(f'Threshold (95th pctl of train): {threshold:.6f}')
print(f'Test anomalies detected: {anomalies.sum()} / {len(anomalies)} ({anomalies.mean()*100:.1f}%)')

---
## Step 7. Result Visualization

### 7-1. Original Sensor Data with Anomaly Markers

원본 온도 데이터와 학습/테스트 경계를 표시합니다.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df['timestamp'], df['temperature'], 'b-', alpha=0.5, linewidth=0.5, label='Temperature')
ax.axvline(x=df['timestamp'].iloc[train_size], color='green', linestyle='--',
           linewidth=2, label='Train/Test boundary')
ax.set_title('Bearing Temperature — Full 90 Days', fontsize=13, fontweight='bold')
ax.set_ylabel('Temperature (C)')
ax.set_xlabel('Date')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7-2. Reconstruction Error (Anomaly Score)

재구성 오차가 임계값(빨간 점선)을 넘으면 **이상으로 판정**합니다.
빨간 영역이 이상으로 감지된 구간입니다.

In [ ]:
test_timestamps = df['timestamp'].iloc[train_size + SEQ_LENGTH : train_size + SEQ_LENGTH + len(test_mse)]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(test_timestamps, test_mse, 'b-', alpha=0.7, linewidth=0.5, label='Reconstruction Error')
ax.axhline(y=threshold, color='red', linestyle='--', linewidth=2,
           label=f'Threshold ({threshold:.4f})')
ax.fill_between(test_timestamps, 0, test_mse,
                where=anomalies[:len(test_timestamps)],
                color='red', alpha=0.3, label='Anomaly Detected!')
ax.set_title('Reconstruction Error — Higher = More Abnormal',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Reconstruction Error (MSE)')
ax.set_xlabel('Date')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7-3. Error Distribution: Train vs Test

학습 데이터(정상)와 테스트 데이터의 재구성 오차 분포를 비교합니다.
테스트 데이터에서 임계값을 넘는 오른쪽 꼬리 부분이 이상 감지 영역입니다.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(train_mse, bins=50, alpha=0.6, color='green', label='Train (Normal)', density=True)
ax.hist(test_mse, bins=50, alpha=0.6, color='blue', label='Test', density=True)
ax.axvline(x=threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold ({threshold:.4f})')
ax.set_title('Reconstruction Error Distribution — Train vs Test',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Reconstruction Error (MSE)')
ax.set_ylabel('Density')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7-4. Multivariate Analysis (LSTM Advantage)

LSTM의 핵심 강점: **3개 센서를 동시에 분석**합니다.
마지막 200시간의 데이터에서 온도, 진동, 전류의 상호 관계를 확인합니다.

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

last_n = 200
t_slice = df['timestamp'].iloc[-last_n:]

l1, = ax1.plot(t_slice, df['temperature'].iloc[-last_n:], 'r-', alpha=0.8, linewidth=1.5, label='Temperature (C)')
l2, = ax1.plot(t_slice, df['vibration'].iloc[-last_n:] * 5, 'b-', alpha=0.8, linewidth=1.5, label='Vibration x5 (mm/s)')
l3, = ax2.plot(t_slice, df['current'].iloc[-last_n:], 'g-', alpha=0.5, linewidth=1, label='Current (A)')

ax1.set_title('LSTM Advantage: Multivariate Analysis (Temp + Vibration + Current)',
              fontsize=13, fontweight='bold')
ax1.set_ylabel('Temperature / Vibration')
ax2.set_ylabel('Current (A)', color='green')
ax1.set_xlabel('Date')
ax1.legend(handles=[l1, l2, l3], fontsize=10, loc='upper left')
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7-5. Reconstruction Example: Normal vs Anomaly

정상 시퀀스와 이상 시퀀스의 복원 결과를 비교합니다.
- **정상**: 원본(파란선)과 복원(빨간 점선)이 거의 일치
- **이상**: 원본과 복원이 크게 차이남 → 높은 재구성 오차

In [ ]:
# Find a normal and anomaly sequence
normal_idx = np.argmin(test_mse)  # lowest error = most normal
anomaly_indices = np.where(anomalies)[0]
anomaly_idx = anomaly_indices[np.argmax(test_mse[anomaly_indices])] if len(anomaly_indices) > 0 else np.argmax(test_mse)

fig, axes = plt.subplots(2, 3, figsize=(15, 7))

for col_i, feat in enumerate(features):
    # Normal
    axes[0, col_i].plot(X_test[normal_idx, :, col_i], 'b-', linewidth=1.5, label='Original')
    axes[0, col_i].plot(test_pred[normal_idx, :, col_i], 'r--', linewidth=1.5, label='Reconstructed')
    axes[0, col_i].set_title(f'Normal — {feat}', fontsize=11)
    axes[0, col_i].legend(fontsize=8)
    axes[0, col_i].grid(True, alpha=0.3)

    # Anomaly
    axes[1, col_i].plot(X_test[anomaly_idx, :, col_i], 'b-', linewidth=1.5, label='Original')
    axes[1, col_i].plot(test_pred[anomaly_idx, :, col_i], 'r--', linewidth=1.5, label='Reconstructed')
    axes[1, col_i].set_title(f'Anomaly — {feat}', fontsize=11)
    axes[1, col_i].legend(fontsize=8)
    axes[1, col_i].grid(True, alpha=0.3)

axes[0, 0].set_ylabel('Normal\n(Low Error)', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Anomaly\n(High Error)', fontsize=11, fontweight='bold')

fig.suptitle(f'Reconstruction Comparison — Normal (MSE={test_mse[normal_idx]:.4f}) vs Anomaly (MSE={test_mse[anomaly_idx]:.4f})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 7-6. Per-Sensor Anomaly Contribution

이상 감지 시 **어떤 센서가 가장 크게 기여**했는지 분석합니다.
이를 통해 "블랙박스" LSTM의 결과를 부분적으로 해석할 수 있습니다.

In [ ]:
# Per-sensor reconstruction error
test_mse_per_sensor = np.mean(np.power(X_test - test_pred, 2), axis=1)  # (n_seq, 3)

fig, ax = plt.subplots(figsize=(8, 5))

# Average error for normal vs anomaly sequences
normal_mask = ~anomalies
normal_err = test_mse_per_sensor[normal_mask].mean(axis=0)
anomaly_err = test_mse_per_sensor[anomalies].mean(axis=0) if anomalies.sum() > 0 else np.zeros(3)

x = np.arange(len(features))
width = 0.35
ax.bar(x - width/2, normal_err, width, label='Normal', color='green', alpha=0.7)
ax.bar(x + width/2, anomaly_err, width, label='Anomaly', color='red', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(features)
ax.set_title('Per-Sensor Reconstruction Error: Normal vs Anomaly',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Mean Reconstruction Error')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print('Anomaly error contribution per sensor:')
if anomaly_err.sum() > 0:
    for feat, err in zip(features, anomaly_err):
        print(f'  {feat:15s}: {err:.4f} ({err/anomaly_err.sum()*100:.1f}%)')

---
## Step 8. Summary

### LSTM Autoencoder 핵심 특징

| 항목 | 내용 |
|------|------|
| **입력** | 3개 센서 × 30시간 윈도우 (다변량) |
| **학습 방식** | 정상 패턴만 학습 (Autoencoder) |
| **이상 감지** | 재구성 오차 > 임계값 → 이상 판정 |
| **임계값** | Train 데이터의 95th percentile |

### 장점
- **다변량**: 온도+진동+전류를 동시에 분석 (Prophet은 단변량만 가능)
- **Autoencoder**: 이상 데이터 없이도 학습 가능 (정상만 있으면 됨)
- **시퀀스 학습**: 30시간 윈도우의 시간적 패턴을 기억

### 단점
- Prophet보다 코드가 복잡
- 하이퍼파라미터 튜닝 필요 (시퀀스 길이, 레이어 수 등)
- "왜 이상인지" 설명이 어려움 (블랙박스) → Per-sensor 분석으로 부분 보완

### 슈레더 적용 권장
- **적합**: 베어링 이상 탐지 (실제 프로젝트에서 사용)
- **부적합**: 장기 트렌드 예측 (→ Prophet 사용)

---

> **다음 단계**: `03_TFT` 폴더에서 Temporal Fusion Transformer를 체험해 보세요. TFT는 LSTM의 다변량 능력 + Prophet의 해석력을 결합한 모델입니다.

In [ ]:
print('=' * 60)
print('  LSTM Autoencoder Simulation Complete!')
print('=' * 60)
print(f'''
  Model: LSTM Autoencoder
  Input: {len(features)} sensors x {SEQ_LENGTH}h window

  Architecture:
    Encoder: LSTM(32) -> LSTM(16)
    Bottleneck: RepeatVector({SEQ_LENGTH})
    Decoder: LSTM(16) -> LSTM(32) -> Dense({len(features)})
    Total params: {model.count_params():,}

  Results:
    Threshold: {threshold:.6f}
    Anomalies detected: {anomalies.sum()} / {len(anomalies)} ({anomalies.mean()*100:.1f}%)
    Final train loss: {history.history["loss"][-1]:.6f}
''')